# 05b — 三维分子构象、3D 描述符与 PubChem API 实战

上一节 `05a` 里所有描述符——`MolWt`、`TPSA`、`BertzCT`、`Kappa2`——都是从 SMILES 的**二维拓扑图**上数出来的：数原子、数环、数键。它们能把水溶解度预测到 $R^2 \approx 0.75$，但剩下那 25% 的误差里，藏着两件二维图永远算不出来的东西：

1. **分子的三维形状**：`C8-BTBT` 的两条正辛基侧链是伸展成一条直线，还是蜷曲成一团？这决定了薄膜里的分子堆积方式、$\pi$–$\pi$ 距离，进而决定载流子迁移率。二维 SMILES 对这两种构象的写法**一模一样**。
2. **晶格能 / 熔点 $T_m$**：从第二讲 `02c` 开始我们就反复引用 Yalkowsky 通用溶解度方程
   $$\log S \approx 0.5 - \log P - 0.01\,(T_m - 25)$$
   其中 $-\log P$ 那一项被 `MolLogP` 很好地覆盖了（`05a` 的 SHAP 蜂群图就是证据），但 $-0.01(T_m - 25)$ 这一项——**破坏晶格所需的能量**——是三维晶体堆积的集体性质，任何单分子 2D 描述符都算不准。

本节分两步补上这两块拼图：
- **前半程（第 1–4 节）**：用 RDKit 的 **ETKDG** 距离几何算法把分子从 SMILES “站起来”变成三维坐标，用 **MMFF94** 力场优化，生成**构象系综（Conformer Ensemble）**并按 Boltzmann 权重加权，再计算 **PMI / NPR 形状三角、Asphericity、回转半径、PBF** 等 3D 描述符，老实检验它们对 $\log S$ 预测到底有多少增益；
- **后半程（第 5–7 节）**：用 **PubChem PUG-REST / PUG-View API** 从全球最大的公开化学数据库批量抓取真实的**实验熔点**，亲手把 Yalkowsky 方程里缺失的 $T_m$ 填进去，验证它能把预测误差压低多少——**这是从第一讲就埋下的伏笔，到这里兑现**。

---

## 0. 环境准备与数据读取辅助函数

In [ ]:
import os
import re
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401  (注册 3d 投影)

import requests

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Descriptors3D, Draw, rdMolDescriptors, rdMolAlign

from sklearn.model_selection import cross_val_score, cross_val_predict, KFold, LeaveOneOut
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 文件夹或当前目录下")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 1. 从 SMILES 到三维坐标：ETKDG 距离几何算法

`Chem.MolFromSmiles("CCO")` 得到的分子对象里**没有任何坐标**，只有“哪个原子连哪个原子”的拓扑图。要得到三维构象，RDKit 走两步：

**第一步：ETKDG 距离几何嵌入（Distance Geometry Embedding）**

**ETKDG**（Experimental-Torsion-angle preference with Knowledge-based basic Distance Geometry，Riniker & Landrum, *J. Chem. Inf. Model.* **55**, 2562, 2015）的思路是：

1. 根据成键关系、键长键角的标准值、环的闭合约束，为**每一对**原子 $(i, j)$ 推出一个距离上下界 $[l_{ij}, u_{ij}]$，构成**界限矩阵（Bounds Matrix）**；
2. 在界限内随机取一个满足三角不等式的距离矩阵，用特征值分解把它嵌入到三维欧氏空间，得到一组初始坐标；
3. **这是 ETKDG 相对于裸距离几何的关键改进**：再叠加一层从剑桥晶体数据库（CSD）统计出来的**二面角（扭转角）偏好**作为惩罚项——例如告诉算法“联苯的两个苯环之间大约扭 $40^\circ$”“酰胺键是平面的”，而不是让它随机乱转。

**第二步：力场优化**

嵌入出来的坐标只是“大致合理”，键长键角都还有误差。再用 **MMFF94（Merck Molecular Force Field）** 或 **UFF（Universal Force Field）** 做几百步能量最小化，把结构收敛到一个局部能量极小点，同时得到该构象的力场能量（单位 kcal/mol）。

> **⚠️ 一个重要的认识**：力场能量**只能在同一个分子的不同构象之间比较**（谁更稳定），**绝不能跨分子比较**（说 A 分子的 MMFF 能量 47 kcal/mol 比 B 分子的 92 kcal/mol “更稳定”是无意义的，因为两者的能量零点不同）。

下面我们把第四讲就反复出现的有机半导体分子 **C8-BTBT**（2,7-二辛基[1]苯并噻吩并[3,2-b][1]苯并噻吩，OTFT 领域的明星材料）站起来：

In [ ]:
SMI_C8BTBT = "CCCCCCCCc1ccc2c(c1)sc1c2sc2cc(CCCCCCCC)ccc21"

mol2d = Chem.MolFromSmiles(SMI_C8BTBT)
print("二维拓扑图：", mol2d.GetNumAtoms(), "个重原子，", mol2d.GetNumBonds(), "根键")
print("此时的构象数（Conformer）：", mol2d.GetNumConformers())

display(Draw.MolToImage(mol2d, size=(470, 200)))

# 1) 加氢——三维构象必须显式包含氢原子，否则立体化学与空间位阻都是错的
mol3d = Chem.AddHs(mol2d)

# 2) ETKDG 嵌入
params = AllChem.ETKDGv3()
params.randomSeed = 42
status = AllChem.EmbedMolecule(mol3d, params)
print(f"\nETKDG 嵌入返回码 = {status}（0 表示成功）")

# 3) MMFF94 力场优化
converged = AllChem.MMFFOptimizeMolecule(mol3d, maxIters=1000)
mp_props = AllChem.MMFFGetMoleculeProperties(mol3d)
ff = AllChem.MMFFGetMoleculeForceField(mol3d, mp_props)
print(f"MMFF 优化收敛码 = {converged}（0 表示已收敛）")
print(f"MMFF 能量 = {ff.CalcEnergy():.2f} kcal/mol")
print(f"加氢后原子数 = {mol3d.GetNumAtoms()}，构象数 = {mol3d.GetNumConformers()}")

conf = mol3d.GetConformer()
print("\n前 5 个原子的三维坐标 (x, y, z) / Angstrom：")
for i in range(5):
    pos = conf.GetAtomPosition(i)
    print(f"  atom {i:2d} ({mol3d.GetAtomWithIdx(i).GetSymbol():2s}): "
          f"{pos.x:7.3f} {pos.y:7.3f} {pos.z:7.3f}")

In [ ]:
def plot_mol_3d(mol, conf_id=0, ax=None, title="", show_h=False):
    """用 matplotlib 把一个 RDKit 构象画成三维骨架图（重原子 + 键）。"""
    conf = mol.GetConformer(conf_id)
    pos = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
    symbols = [a.GetSymbol() for a in mol.GetAtoms()]
    palette = {"C": "#3a3a3a", "S": "#d4a017", "O": "#c0392b", "N": "#2874a6", "H": "#bbbbbb"}

    if ax is None:
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(111, projection="3d")

    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        if not show_h and ("H" in (symbols[i], symbols[j])):
            continue
        ax.plot(*zip(pos[i], pos[j]), color="#888888", lw=1.3, zorder=1)

    keep = [i for i, s in enumerate(symbols) if show_h or s != "H"]
    ax.scatter(pos[keep, 0], pos[keep, 1], pos[keep, 2],
               c=[palette.get(symbols[i], "#7f8c8d") for i in keep],
               s=[70 if symbols[i] == "S" else 36 for i in keep],
               depthshade=True, zorder=2)

    # 等比例坐标轴，否则分子会被拉变形
    span = (pos[keep].max(axis=0) - pos[keep].min(axis=0)).max() / 2
    mid = (pos[keep].max(axis=0) + pos[keep].min(axis=0)) / 2
    ax.set_xlim(mid[0] - span, mid[0] + span)
    ax.set_ylim(mid[1] - span, mid[1] + span)
    ax.set_zlim(mid[2] - span, mid[2] + span)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
    return ax

plot_mol_3d(mol3d, 0, title="C8-BTBT: one ETKDG + MMFF optimized conformer")
plt.show()

---

## 2. 构象系综：一个分子不止一个形状

上面那张图只是 C8-BTBT **某一个**局部能量极小点。它有两条正辛基链，每条链上有 7 根可旋转单键——粗略估计构象数在 $3^{14} \approx 5 \times 10^{6}$ 量级。室温下分子其实在这些构象之间快速翻转，任何实验测到的性质都是整个**构象系综（Conformer Ensemble）**的 Boltzmann 加权平均。

标准工作流是：

1. `EmbedMultipleConfs(mol, numConfs=N, params=ETKDGv3())` 一次生成 $N$ 个初始构象（随机种子各不相同）；
2. `MMFFOptimizeMoleculeConfs(mol)` 批量优化，返回每个构象的 `(收敛码, 能量)`；
3. 按能量排序，用 **RMSD（均方根偏差）** 去掉彼此重复的构象；
4. 按 Boltzmann 因子 $w_i \propto \exp(-\Delta E_i / k_B T)$ 加权（室温下 $k_B T \approx 0.593$ kcal/mol）。

$N$ 该取多少？一个被广泛引用的经验规则（Ebejer et al., *J. Chem. Inf. Model.* **52**, 1146, 2012）按可旋转键数 `NumRotatableBonds` 分档：$\le 7$ 根取 50 个，8–12 根取 200 个，更多取 300 个。本节为了课堂速度只取 30 个。

In [ ]:
N_CONFS = 30
KT_298 = 0.593          # k_B * T，单位 kcal/mol（T = 298 K）

mol_ens = Chem.AddHs(Chem.MolFromSmiles(SMI_C8BTBT))
params = AllChem.ETKDGv3()
params.randomSeed = 42
params.pruneRmsThresh = -1.0        # 先不剪枝，后面手动按 RMSD 去重

t0 = time.time()
cids = AllChem.EmbedMultipleConfs(mol_ens, numConfs=N_CONFS, params=params)
results = AllChem.MMFFOptimizeMoleculeConfs(mol_ens, maxIters=800)
energies = np.array([e for _conv, e in results])
print(f"生成并优化 {len(cids)} 个构象，用时 {time.time() - t0:.1f} s")

order = np.argsort(energies)
E_rel = energies - energies.min()         # 相对于最稳定构象的能量差
weights = np.exp(-E_rel / KT_298)
weights /= weights.sum()
n_eff = 1.0 / np.sum(weights ** 2)        # 有效构象数：真正参与分布的有几个

print(f"MMFF 能量范围: {energies.min():.2f} ~ {energies.max():.2f} kcal/mol"
      f"（跨度 {E_rel.max():.2f} kcal/mol）")
print(f"最稳定构象的 Boltzmann 权重 = {weights[order[0]]:.3f}")
print(f"有效构象数 n_eff = 1 / sum(w^2) = {n_eff:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
axes[0].bar(range(len(E_rel)), np.sort(E_rel), color=C.navy)
axes[0].set_xlabel("Conformer rank (sorted by energy)")
axes[0].set_ylabel("Relative MMFF energy  [kcal/mol]")
axes[0].set_title("Conformer Energy Ladder")
axes[0].axhline(KT_298, color=C.crimson, ls="--", lw=1.4, label="kT at 298 K = 0.59")
axes[0].legend(fontsize=8)

axes[1].bar(range(len(weights)), np.sort(weights)[::-1], color=C.gold)
axes[1].set_xlabel("Conformer rank")
axes[1].set_ylabel("Boltzmann weight")
axes[1].set_title(f"Boltzmann Populations at 298 K  (n_eff = {n_eff:.1f})")
plt.show()

In [ ]:
# 用 RMSD 判断“两个构象是不是其实长得一样”
mol_noH = Chem.RemoveHs(mol_ens)
best, second, worst = int(order[0]), int(order[1]), int(order[-1])

rms_12 = rdMolAlign.GetBestRMS(mol_noH, mol_noH, prbId=second, refId=best)
rms_1N = rdMolAlign.GetBestRMS(mol_noH, mol_noH, prbId=worst,  refId=best)
print(f"最稳定构象 vs 次稳定构象 RMSD = {rms_12:.3f} Angstrom")
print(f"最稳定构象 vs 最高能构象 RMSD = {rms_1N:.3f} Angstrom")

# 贪心去重：RMSD < 0.5 A 视为同一个构象
RMS_THRESH = 0.5
unique = []
for cid in order:
    cid = int(cid)
    if all(rdMolAlign.GetBestRMS(mol_noH, mol_noH, prbId=cid, refId=u) > RMS_THRESH
           for u in unique):
        unique.append(cid)
print(f"\n按 RMSD < {RMS_THRESH} A 去重后：{len(cids)} -> {len(unique)} 个独立构象")

fig = plt.figure(figsize=(12, 4))
for k, cid in enumerate([best, second, worst]):
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    plot_mol_3d(mol_ens, cid, ax=ax,
                title=f"conf {cid}:  dE = {E_rel[cid]:.2f} kcal/mol,  w = {weights[cid]:.3f}")
plt.show()

三张图并排放在一起，差别一目了然：最稳定构象的两条辛基链**伸展**成近乎平面的 all-anti 构型（这正是 C8-BTBT 能在薄膜里形成层状堆积、获得高迁移率的结构基础），而最高能构象的链**蜷曲**折叠了起来。二维 SMILES 对这两者的描述完全相同。

另外注意去重那一行的结果：C8-BTBT 的 30 个构象**一个都没有被合并**——对于一个有 14 根可旋转键的分子，30 次嵌入连构象空间的皮毛都没摸到，彼此之间自然没有重复。这反过来说明：**柔性越大的分子，`numConfs` 就必须开得越大**，否则你算出来的"Boltzmann 加权 3D 描述符"只是对一小撮随机构象求了平均。

### 🏋️ 练习 05b-1：构象数与可旋转键数的关系

对下面四个可旋转键数递增的分子：

- `Benzene`: `"c1ccccc1"`（0 根）
- `Biphenyl`: `"c1ccc(-c2ccccc2)cc1"`（1 根）
- `Dibutyl phthalate`: `"CCCCOC(=O)c1ccccc1C(=O)OCCCC"`（约 9 根）
- `C8-BTBT`: 上面的 `SMI_C8BTBT`（14 根）

1. 用 `Descriptors.NumRotatableBonds` 读出每个分子的可旋转键数；
2. 对每个分子用 `EmbedMultipleConfs(numConfs=50)` + `MMFFOptimizeMoleculeConfs` 生成系综，并用 `RMSD > 0.5 A` 的贪心去重算出**独立构象数**；
3. 画出“可旋转键数 vs 独立构象数”的散点图，并报告每个分子的能量跨度 `E_rel.max()`。

> **提示**：苯和联苯会返回很少的独立构象——这正说明刚性稠环体系用单构象近似是安全的，而柔性侧链体系必须做系综。

In [ ]:
# 在下方填写你的代码：


---

## 3. 3D 描述符：把形状变成数字

有了三维坐标，就能计算一类二维图完全无法企及的描述符。它们大多基于**主惯性矩（Principal Moments of Inertia, PMI）** $I_1 \le I_2 \le I_3$——把分子看成一个刚体，这三个数描述了它绕三根主轴转动的惯性。

| 描述符 | 定义 / 取值范围 | 物理含义 |
|---|---|---|
| **NPR1, NPR2** | $I_1/I_3$ 与 $I_2/I_3$，均 $\in [0, 1]$ | **归一化主惯性比**，最常用的一对形状坐标 |
| **Asphericity** | $0$（球）$\to 1$（线） | 偏离球形的程度 |
| **SpherocityIndex** | $0 \to 1$（完美球） | 球形度 |
| **RadiusOfGyration** | 单位 Å | 回转半径，刻画分子“摊开”得多大 |
| **PBF** | Plane of Best Fit，单位 Å | 全部原子到最佳拟合平面的平均距离；完全平面的稠环分子接近 0 |
| **InertialShapeFactor** | $I_2 / (I_1 I_3)$ | 另一种形状不变量 |

**NPR 形状三角（PMI Triangle）** 是这类描述符最著名的可视化：把每个分子画在 $(\text{NPR1}, \text{NPR2})$ 平面上，所有分子都落在一个三角形内，三个顶点分别是：

- $(0, 1)$ —— **棒状（rod）**：如乙炔、直链烷烃；
- $(0.5, 0.5)$ —— **盘状（disc）**：如苯、并五苯等平面稠环；
- $(1, 1)$ —— **球状（sphere）**：如金刚烷、立方烷、$\text{C}_{60}$。

这张图在药物化学里被用来评价化合物库的“形状多样性”，在有机半导体里则直接关联分子堆积模式——盘状分子才容易形成 $\pi$–$\pi$ 柱状或鱼骨堆积。

In [ ]:
SHAPE_PROBES = {
    "Acetylene (rod)":     "C#C",
    "Hexane (rod-ish)":    "CCCCCC",
    "Benzene (disc)":      "c1ccccc1",
    "Anthracene (disc)":   "c1ccc2cc3ccccc3cc2c1",
    "Pentacene (disc)":    "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "Adamantane (sphere)": "C1C2CC3CC1CC(C2)C3",
    "Cubane (sphere)":     "C12C3C4C1C5C4C3C25",
    "Triphenylamine":      "c1ccc(N(c2ccccc2)c2ccccc2)cc1",
    "C8-BTBT":             SMI_C8BTBT,
}

DESC_3D = {
    "NPR1":                Descriptors3D.NPR1,
    "NPR2":                Descriptors3D.NPR2,
    "Asphericity":         Descriptors3D.Asphericity,
    "SpherocityIndex":     Descriptors3D.SpherocityIndex,
    "RadiusOfGyration":    Descriptors3D.RadiusOfGyration,
    "Eccentricity":        Descriptors3D.Eccentricity,
    "InertialShapeFactor": Descriptors3D.InertialShapeFactor,
    "PBF":                 rdMolDescriptors.CalcPBF,
}

def embed_best_conformer(smi, n_confs=10, seed=42, max_iters=600):
    """生成 n_confs 个构象、MMFF 优化后返回能量最低的那一个（单构象近似）。"""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    ps = AllChem.ETKDGv3()
    ps.randomSeed = seed
    cids_ = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=ps)
    if len(cids_) == 0:
        return None
    try:
        res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=max_iters)
    except Exception:
        return None
    best_cid = int(np.argmin([e for _c, e in res]))
    keep = Chem.Mol(mol)
    keep.RemoveAllConformers()
    keep.AddConformer(mol.GetConformer(best_cid), assignId=True)
    return keep

shape_rows = []
for name, smi in SHAPE_PROBES.items():
    m = embed_best_conformer(smi)
    if m is None:
        continue
    row = {"molecule": name}
    row.update({k: fn(m) for k, fn in DESC_3D.items()})
    shape_rows.append(row)

shape_df = pd.DataFrame(shape_rows).set_index("molecule")
print(shape_df.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 6))

# 形状三角的三条边
ax.plot([0.0, 0.5], [1.0, 0.5], color="#999999", lw=1.2)   # rod  -> disc
ax.plot([0.5, 1.0], [0.5, 1.0], color="#999999", lw=1.2)   # disc -> sphere
ax.plot([0.0, 1.0], [1.0, 1.0], color="#999999", lw=1.2)   # rod  -> sphere

ax.scatter(shape_df["NPR1"], shape_df["NPR2"], s=90, color=C.crimson, zorder=3)
for name, r in shape_df.iterrows():
    ax.annotate(name, (r["NPR1"], r["NPR2"]), textcoords="offset points",
                xytext=(7, 4), fontsize=8)

for (x, y), lab in [((0.0, 1.0), "ROD"), ((0.5, 0.5), "DISC"), ((1.0, 1.0), "SPHERE")]:
    ax.annotate(lab, (x, y), textcoords="offset points", xytext=(-8, -24),
                fontsize=11, fontweight="bold", color=C.navy)

ax.set_xlabel("NPR1 = I1 / I3")
ax.set_ylabel("NPR2 = I2 / I3")
ax.set_xlim(-0.08, 1.12)
ax.set_ylim(0.42, 1.08)
ax.set_title("PMI Shape Triangle of Representative Molecules")
plt.show()

并苯家族（苯 → 蒽 → 并五苯）沿着“盘状”顶点附近排成一列，且随着环数增加越来越贴近 rod–disc 那条边——它们是**完美的平面盘状分子**，`PBF` 几乎为 0。这正是并苯类材料能形成 herringbone（鱼骨）堆积、获得高迁移率的几何根源。金刚烷和立方烷则牢牢占据球状顶点。

### 🏋️ 练习 05b-2：构象系综会让 3D 描述符飘多少？

3D 描述符的致命弱点是**构象依赖**：同一个分子的不同构象，`RadiusOfGyration` 可以差出 2 Å。

对 `Dibutyl phthalate`（`"CCCCOC(=O)c1ccccc1C(=O)OCCCC"`）：
1. 生成 50 个构象并 MMFF 优化，对**每一个**构象计算 `RadiusOfGyration`、`Asphericity` 和 `NPR1`；
2. 画出这三个描述符在系综中的分布直方图，报告它们的**最小值、最大值与标准差**；
3. 分别计算三者的**简单平均**与 **Boltzmann 加权平均**（用第 2 节的 `KT_298`），比较两者的差异。

> **思考**：如果一篇论文只说“用 RDKit 算了 3D 描述符”而不交代构象是怎么选的，你能复现它的结果吗？

In [ ]:
# 在下方填写你的代码：


---

## 4. 老实检验：3D 描述符对 $\log S$ 预测有多少增益？

3D 描述符听上去更“物理”，但它贵得多：每个分子要做 ETKDG 嵌入 + 力场优化，比算 `MolWt` 慢三四个数量级。它值吗？

我们做一个干净的对照实验：从 AqSolDB 里取 800 个中等大小、柔性不太高的分子（这样单构象近似还算合理），分别用

- **2D only**：10 个经典 2D 描述符；
- **3D only**：8 个 3D 描述符；
- **2D + 3D**：全部 18 个；

训练随机森林，比较五折交叉验证 $R^2$。（下面这格要为 800 个分子做构象嵌入，约需 30–90 秒。）

In [ ]:
df_sol = pd.read_csv(find_file("solubility.csv"))
pool = df_sol[(df_sol["MolWt"].between(60, 400)) & (df_sol["NumRotatableBonds"] < 12)]
pool = pool.sample(800, random_state=0).reset_index(drop=True)

D2_COLS = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
           "NumRotatableBonds", "NumAromaticRings", "RingCount", "BertzCT", "BalabanJ"]

t0 = time.time()
rows3d, ok_idx = [], []
for i, smi in enumerate(pool["SMILES"]):
    m = embed_best_conformer(smi, n_confs=1, seed=42, max_iters=500)
    if m is None:
        continue
    try:
        rows3d.append({k: fn(m) for k, fn in DESC_3D.items()})
    except Exception:
        continue
    ok_idx.append(i)

print(f"为 {len(ok_idx)} / {len(pool)} 个分子成功生成三维构象，用时 {time.time() - t0:.1f} s")

pool_ok = pool.loc[ok_idx].reset_index(drop=True)
X_3d = pd.DataFrame(rows3d).reset_index(drop=True)
X_2d = pool_ok[D2_COLS].reset_index(drop=True)
X_23 = pd.concat([X_2d, X_3d], axis=1)
y_sol = pool_ok["Solubility"].values

In [ ]:
cv5 = KFold(n_splits=5, shuffle=True, random_state=0)
bench = []
for name, X_ in [("2D only (10)", X_2d), ("3D only (8)", X_3d), ("2D + 3D (18)", X_23)]:
    s = cross_val_score(RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
                        X_, y_sol, cv=cv5, scoring="r2")
    bench.append({"features": name, "n": X_.shape[1],
                  "cv_r2_mean": round(s.mean(), 4), "cv_r2_std": round(s.std(), 4)})

bench = pd.DataFrame(bench)
print(bench.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(bench["features"], bench["cv_r2_mean"], yerr=bench["cv_r2_std"],
       capsize=5, color=[C.navy, C.gold, C.crimson], width=0.55)
for i, r in bench.iterrows():
    ax.text(i, r["cv_r2_mean"] + r["cv_r2_std"] + 0.015, f"{r['cv_r2_mean']:.3f}",
            ha="center", fontsize=10)
ax.set_ylabel("5-fold CV R2")
ax.set_ylim(0, 0.9)
ax.set_title("Do 3D Descriptors Help Predicting logS?")
plt.show()

**结果是一记有益的清醒剂**：3D 描述符单独使用时远不如 2D（它们完全没有编码极性、氢键这些对水溶解度至关重要的信息），加进 2D 之后带来的提升也只有一两个百分点——而计算代价增加了几千倍。

这不是说 3D 描述符没用，而是说**要用在对的地方**：

| 适合 3D 描述符的性质 | 原因 |
|---|---|
| 蛋白–配体结合亲和力 | 结合口袋的形状互补性是决定性的 |
| 载流子迁移率、$\pi$–$\pi$ 堆积 | 平面度（`PBF`）、分子长宽比直接决定堆积模式 |
| 液晶相行为、熔点 | 形状各向异性决定能否形成有序相 |
| 膜渗透性、口服生物利用度 | 3D 极性表面积比 2D TPSA 更准 |

| 不太需要 3D 的性质 | 原因 |
|---|---|
| 水溶解度 $\log S$ | 主要由极性/疏水平衡决定，2D 描述符已捕捉 |
| $\log P$ | 本身就是原子贡献加和模型 |
| 简单的毒性/活性分类 | 子结构指纹通常已经够用 |

> **📌 工程准则**：先用便宜的 2D 描述符做基线，**只有当某个物理机制明确依赖三维形状时**才付出 3D 的代价，并且一定要说明构象是怎么选的（单构象？能量最低？Boltzmann 加权？）。

---

## 5. PubChem PUG-REST：从数据库里“要”数据

到目前为止，本课程所有数据都来自预先下载好的 CSV。但真实课题的第一步往往是：**我有一张 50 个化合物的名单，我需要它们的实验性质**。

**PubChem** 是 NIH 维护的全球最大公开化学数据库（超过 1 亿个化合物），它提供两套 HTTP API：

| 接口 | 用途 | URL 前缀 |
|---|---|---|
| **PUG-REST** | 查**计算性质**与标识符（分子式、分子量、XLogP、SMILES、InChI、CID） | `https://pubchem.ncbi.nlm.nih.gov/rest/pug` |
| **PUG-View** | 查**实验性质注释**（熔点、沸点、溶解度、光谱），带文献出处 | `https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound` |

PUG-REST 的 URL 遵循一个非常规整的三段式语法：

```
.../rest/pug / <输入域> / <操作> / <输出格式>

compound/name/aspirin  /  property/MolecularWeight,XLogP  /  JSON
compound/cid/2244      /  property/MolecularFormula       /  CSV
compound/smiles/CCO    /  cids                            /  TXT
```

> **⚠️ 使用礼仪（这是硬性要求，不是建议）**：PubChem 限制**每秒不超过 5 次请求**，超限会返回 `503`。批量抓取时务必 `time.sleep(0.2)` 以上，并且**能批量就不要循环单查**——`compound/cid/2244,931,8418/property/...` 一次可以要回上百个 CID 的结果。

> **⚠️ 关于属性名**：PubChem 会不定期调整 property 名称（例如 `CanonicalSMILES` 近年被 `SMILES` / `ConnectivitySMILES` 取代）。写批量脚本时最好**只请求少数几个稳定属性**，并对返回的列名做容错处理。

In [ ]:
PUG_REST = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
PUG_VIEW = "https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound"
PAUSE    = 0.25        # 每次请求后的礼貌等待，满足 <= 5 req/s 的限制

def pug_get(url, params=None, timeout=25, retries=3):
    """带重试与限速的 GET；失败时返回 None 而不是抛异常，便于批量抓取时跳过。"""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            time.sleep(PAUSE)
            if r.status_code == 200:
                return r
            if r.status_code == 404:         # PubChem 里没有这个条目，重试也没用
                return None
            time.sleep(1.5 * (attempt + 1))  # 503 / 限流：退避后重试
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
    return None

# —— 演示 1：按名称查 CID ——
r = pug_get(f"{PUG_REST}/compound/name/anthracene/cids/JSON")
print("按名称查 CID:", r.json() if r else "请求失败（可能是网络不通）")

# —— 演示 2：按 CID 查计算性质 ——
r = pug_get(f"{PUG_REST}/compound/cid/8418/property/MolecularFormula,MolecularWeight,XLogP,TPSA/JSON")
if r:
    print("\n按 CID 查性质:")
    print(json.dumps(r.json()["PropertyTable"]["Properties"][0], indent=2, ensure_ascii=False))

# —— 演示 3：按 SMILES 反查 CID（结构检索）——
r = pug_get(f"{PUG_REST}/compound/smiles/{requests.utils.quote('c1ccc2cc3ccccc3cc2c1')}/cids/TXT")
print("\n按 SMILES 反查 CID:", r.text.strip().splitlines()[:3] if r else "请求失败")

In [ ]:
# —— 批量抓取：一次请求拿回几十个化合物的性质 ——
PROBE_CIDS = [241, 931, 8418, 995, 31423, 9171, 9142, 6853, 8030, 7047]  # 苯、萘、蒽、菲……

def fetch_properties(cids_, props=("MolecularFormula", "MolecularWeight", "XLogP", "TPSA"),
                     chunk=50):
    """分批调用 PUG-REST 的 property 接口，返回一张 DataFrame。"""
    frames = []
    for i in range(0, len(cids_), chunk):
        part = ",".join(str(c) for c in cids_[i:i + chunk])
        r = pug_get(f"{PUG_REST}/compound/cid/{part}/property/{','.join(props)}/JSON")
        if r is None:
            continue
        frames.append(pd.DataFrame(r.json()["PropertyTable"]["Properties"]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

props_df = fetch_properties(PROBE_CIDS)
if len(props_df):
    print(props_df.to_string(index=False))
else:
    print("网络不可用，跳过在线抓取（后面的分析会自动改用随课程打包的本地数据）")

---

## 6. PUG-View：抓取实验熔点并解析自由文本

实验性质比计算性质难拿得多：PubChem 把它们存成**带文献出处的自由文本注释**，形态五花八门——`"135 °C"`、`"216-218 °C"`、`"80.2 deg C"`、`"300 °F"`、`"MP: 122.4 °C (lit.)"`。要把它们变成一列干净的浮点数，必须自己写解析器。

我们的策略：

1. 请求 `pug_view/data/compound/<CID>/JSON?heading=Melting+Point`；
2. **递归**遍历返回的嵌套 JSON，把其中所有字符串收集起来（PubChem 的注释层级很深，写死路径会很脆）；
3. 用正则匹配 `数值 [区间] [单位]`，区间取中点，华氏转摄氏，再用常识范围过滤掉明显不是温度的数字；
4. 同一个化合物常有多条来源（Merck Index、NTP、OSHA……），**取中位数**比取第一条稳健得多。

这就是真实数据工程的日常：**大半的时间花在把别人的文本变成自己的数字上**。

In [ ]:
MP_PATTERN = re.compile(
    r"(-?\d+(?:\.\d+)?)\s*(?:-|to|–)?\s*(-?\d+(?:\.\d+)?)?\s*°?\s*([CF])"
)

def iter_strings(obj):
    """递归遍历任意嵌套的 JSON，产出其中所有字符串。"""
    if isinstance(obj, dict):
        for v in obj.values():
            yield from iter_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_strings(v)
    elif isinstance(obj, str):
        yield obj

def parse_melting_point(text):
    """从一条自由文本里解析摄氏熔点；解析不出来返回 None。"""
    if "http" in text or len(text) > 60:        # 跳过参考文献与长段落
        return None
    m = MP_PATTERN.search(text.replace("\u00b0", " ").replace("deg", " "))
    if m is None:
        return None
    lo = float(m.group(1))
    hi = float(m.group(2)) if m.group(2) else lo
    t = (lo + hi) / 2
    if m.group(3) == "F":                       # 华氏 -> 摄氏
        t = (t - 32) * 5 / 9
    return t if -180 < t < 600 else None        # 常识范围过滤

def fetch_melting_point(cid):
    """抓取并解析某个 CID 的实验熔点（取所有来源的中位数）。"""
    r = pug_get(f"{PUG_VIEW}/{cid}/JSON", params={"heading": "Melting Point"})
    if r is None:
        return None
    vals = []
    for s in iter_strings(r.json()):
        t = parse_melting_point(s)
        if t is not None:
            vals.append(t)
    return float(np.median(vals)) if vals else None

# 在几个熟悉的分子上试一试（文献值：萘 80.2、蒽 216、菲 100、芘 151、咖啡因 236 ℃）
DEMO = {"naphthalene": 931, "anthracene": 8418, "phenanthrene": 995,
        "pyrene": 31423, "caffeine": 2519}
print(f"{'molecule':<16s}{'CID':>8s}{'Tm / degC':>14s}")
for name, cid in DEMO.items():
    tm = fetch_melting_point(cid)
    print(f"{name:<16s}{cid:>8d}{(f'{tm:.1f}' if tm is not None else 'n/a'):>14s}")

---

## 7. 补全伏笔：把 $T_m$ 填进 Yalkowsky 方程

现在我们有了抓取熔点的能力。为了让课堂不受网络状况影响，课程随数据集打包了一张 **`data/melting_points.csv`**——它正是用上面这套代码对 79 个常见有机分子（并苯、杂环、酚类、羧酸、农药、甾体激素等）批量抓取 PubChem 实验熔点，并按**规范化 SMILES** 与 AqSolDB 的实测 $\log S$ 对齐后的结果。

> **为什么要按规范化 SMILES（canonical SMILES）对齐，而不是按名字？** 同一个分子在 PubChem 叫 `naphthalen-1-ol`，在 AqSolDB 叫 `1-naphthol`，按名字匹配会大量漏掉。把两边的 SMILES 都用 `Chem.MolToSmiles(Chem.MolFromSmiles(smi))` 过一遍 RDKit 的规范化算法，得到的字符串就是唯一的——**这是跨数据库合并化学数据的标准做法**。

Yalkowsky 通用溶解度方程（General Solubility Equation, GSE）：

$$\log S \approx 0.5 - \log P - 0.01\,(T_m - 25)$$

它由两项构成：$-\log P$ 是**溶剂化项**（把分子从自身液相搬进水里），$-0.01(T_m - 25)$ 是**晶格破坏项**（把分子从晶体里拆出来；熔点越高说明晶格越稳，越难溶）。下面我们正面检验：加上 $T_m$ 到底能把误差压低多少。

In [ ]:
mp_df = pd.read_csv(find_file("melting_points.csv"))
print(f"本地熔点数据集: {len(mp_df)} 个分子")
print(mp_df.head(8).to_string(index=False))
print(f"\n熔点范围: {mp_df['Tm_C'].min():.1f} ~ {mp_df['Tm_C'].max():.1f} degC；"
      f"logS 范围: {mp_df['logS'].min():.2f} ~ {mp_df['logS'].max():.2f}")

In [ ]:
# 可选：抽样校验本地熔点与此刻 PubChem 线上值是否一致
CHECK_N = 5
check = mp_df.sample(CHECK_N, random_state=1)
print(f"{'name':<22s}{'local Tm':>10s}{'live Tm':>10s}")
for _, row in check.iterrows():
    live = fetch_melting_point(int(row["cid"]))
    live_s = f"{live:.1f}" if live is not None else "n/a"
    print(f"{row['name']:<22s}{row['Tm_C']:>10.1f}{live_s:>10s}")
print("\n（显示 n/a 通常只是当前网络不通，不影响下面使用本地数据的分析）")

In [ ]:
logP = mp_df["MolLogP"].values
Tm   = mp_df["Tm_C"].values
logS = mp_df["logS"].values

gse_full = 0.5 - logP - 0.01 * (Tm - 25)      # 完整 Yalkowsky 方程
gse_noTm = 0.5 - logP                          # 砍掉晶格项

def report(pred, label):
    rmse = np.sqrt(np.mean((pred - logS) ** 2))
    mae  = np.mean(np.abs(pred - logS))
    r    = np.corrcoef(pred, logS)[0, 1]
    print(f"{label:<36s} RMSE = {rmse:.3f}   MAE = {mae:.3f}   r = {r:.3f}")
    return rmse

rmse_no = report(gse_noTm, "GSE without Tm  (0.5 - logP)")
rmse_tm = report(gse_full, "GSE with    Tm  (full Yalkowsky)")
print(f"\n加入实验熔点后 RMSE 下降 {rmse_no - rmse_tm:.3f} 个对数单位"
      f"（相对改善 {100 * (rmse_no - rmse_tm) / rmse_no:.1f}%）")

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.6), sharex=True, sharey=True)
for ax, pred, title, rmse in [
    (axes[0], gse_noTm, "Without melting point", rmse_no),
    (axes[1], gse_full, "Full Yalkowsky GSE (with Tm)", rmse_tm),
]:
    sc = ax.scatter(logS, pred, c=Tm, cmap="plasma", s=42, alpha=0.9, edgecolor="none")
    lims = [-9.5, 1.5]
    ax.plot(lims, lims, "--", color="#888888", lw=1.2)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("Measured logS")
    ax.set_title(f"{title}\nRMSE = {rmse:.3f}", fontsize=10)
axes[0].set_ylabel("Predicted logS")
plt.colorbar(sc, ax=axes, shrink=0.85, label="Melting point / degC")
plt.show()

左图里**高熔点分子（亮黄色点）系统性地落在对角线上方**——不含 $T_m$ 的公式一律高估了它们的溶解度，因为它完全不知道这些分子的晶格有多稳。右图加上晶格项后，黄点被整体拉回对角线，误差分布也不再有明显的颜色梯度。

**这正是第二讲 `02c` 里“模型漏报高对称性难溶分子”现象的物理解释**：那些被随机森林判成“应该挺好溶”却实际难溶的分子，正是熔点很高的 brick-dust（砖头粉）型化合物，而我们当时的 2D 描述符里根本没有熔点这一维。

### 7.1 让数据自己拟合系数：Yalkowsky 的三个常数对不对？

GSE 里的三个常数 $0.5$、$-1$、$-0.01$ 是 Yalkowsky 在 1980 年代从大量实验数据里总结出来的。现在我们有 79 个分子，可以用最小二乘让数据自己说话：

$$\log S = \beta_0 + \beta_1 \log P + \beta_2 T_m$$

把 GSE 展开成同样的形式是 $\log S = (0.5 + 0.25) - 1 \cdot \log P - 0.01\,T_m$，所以理论值应当是 $(\beta_0, \beta_1, \beta_2) \approx (0.75,\ -1,\ -0.01)$。

In [ ]:
X_gse = mp_df[["MolLogP", "Tm_C"]].values
lr = LinearRegression().fit(X_gse, logS)

print("数据拟合结果：")
print(f"  logS = {lr.intercept_:+.3f} {lr.coef_[0]:+.3f} * logP {lr.coef_[1]:+.4f} * Tm")
print("Yalkowsky 理论值：")
print("  logS =  +0.750  -1.000 * logP  -0.0100 * Tm")

# 用留一交叉验证（LOO）公平比较“有没有 Tm”
for cols, label in [(["MolLogP"], "logP only"), (["MolLogP", "Tm_C"], "logP + Tm")]:
    pred = cross_val_predict(LinearRegression(), mp_df[cols].values, logS, cv=LeaveOneOut())
    rmse = np.sqrt(np.mean((pred - logS) ** 2))
    r2   = 1 - np.sum((pred - logS) ** 2) / np.sum((logS - logS.mean()) ** 2)
    print(f"\nLOO 交叉验证  {label:<12s}: RMSE = {rmse:.3f}, R2 = {r2:.3f}")

拟合出来的斜率与 Yalkowsky 四十年前给出的 $-1$ 和 $-0.01$ 处在同一量级且方向一致——**一个 1980 年代的半经验方程，在今天的数据上依然立得住**。留一交叉验证则显示：仅仅加入一列熔点，$R^2$ 就有可观提升，RMSE 下降了 0.2 个对数单位以上。

这也回应了第七讲 `07a` 里留下的那个问题（“梯度下降拟合 $\log S$–$\log P$ 得到的斜率约 $-0.41$，为什么比 Yalkowsky 的 $-1$ 平缓得多？”）：在全量 AqSolDB 上，大量高熔点分子的晶格效应被**吸收进了残差**，把回归斜率压平了；一旦把 $T_m$ 显式放进模型，$\log P$ 的系数立刻朝理论值靠拢。

### 🏋️ 练习 05b-3：把熔点加进机器学习模型

`data/melting_points.csv` 里有 79 个分子的 `SMILES`、`Tm_C` 和 `logS`。

1. 用 `05a` 的方式为这 79 个分子计算一组 2D 描述符（至少包含 `MolWt`、`MolLogP`、`TPSA`、`NumAromaticRings`、`NumHDonors`、`BertzCT`）；
2. 用留一交叉验证（`LeaveOneOut`）比较三个随机森林（`n_estimators=300, random_state=0`）：
   - **A**：只用 2D 描述符；
   - **B**：2D 描述符 + `Tm_C`；
   - **C**：只用 `MolLogP` 和 `Tm_C`（即 Yalkowsky 的两个变量）；
3. 报告三者的 LOO RMSE 与 $R^2$，并画出 B 模型的预测–实测散点图；
4. **讨论**：模型 C 只用两个变量，表现与用了一堆描述符的 B 差多少？这说明什么？

> **提示**：79 个样本非常少，`KFold(5)` 的折间方差会很大，所以这里用 LOO；随机森林在这种小样本上会有一定过拟合，属于预期内的现象。

In [ ]:
# 在下方填写你的代码：


---

## 小结

| 核心概念 | 方法与实现 | 材料与化学意义 |
|---|---|---|
| **ETKDG 距离几何嵌入** | `AllChem.EmbedMolecule(mol, ETKDGv3())`，**必须先 `AddHs`** | 界限矩阵 + CSD 统计的二面角偏好，把拓扑图变成合理的三维坐标 |
| **MMFF94 力场优化** | `AllChem.MMFFOptimizeMolecule` | 收敛到局部能量极小点；能量**只能在同分子的不同构象间比较** |
| **构象系综与 Boltzmann 权重** | `EmbedMultipleConfs` + `GetBestRMS` 去重 + $w_i \propto e^{-\Delta E_i / k_B T}$ | C8-BTBT 的辛基链伸展构象最稳定，正是其层状堆积、高迁移率的几何基础 |
| **PMI 形状三角（NPR1/NPR2）** | `Descriptors3D.NPR1 / NPR2` | 并苯类聚在“盘状”顶点（`PBF` 接近 0），金刚烷、立方烷在“球状”顶点 |
| **3D 描述符的收益边界** | 2D vs 3D vs 2D+3D 的五折 CV 对照 | 对 $\log S$ 增益很小；真正值得付出 3D 代价的是结合亲和力、堆积与迁移率 |
| **PUG-REST 批量抓取** | `compound/cid/<多个CID>/property/<属性>/JSON`，不超过 5 req/s | 能批量就不要循环单查；带重试与退避的 `pug_get` 是生产级写法 |
| **PUG-View 实验性质解析** | `heading=Melting Point` + 递归遍历 + 正则 + 多来源取中位数 | 实验数据是带文献出处的自由文本，解析器本身就是数据工程的主体 |
| **规范化 SMILES 跨库对齐** | `Chem.MolToSmiles(Chem.MolFromSmiles(s))` | PubChem 的 `naphthalen-1-ol` 与 AqSolDB 的 `1-naphthol` 靠它才能对上 |
| **Yalkowsky GSE 伏笔兑现** | $\log S \approx 0.5 - \log P - 0.01(T_m - 25)$ | 加入实验熔点后 RMSE 显著下降，拟合系数与 1980 年代的理论值方向、量级一致 |

### 第五讲全景回顾

| 问题 | `05a` 的回答 | `05b` 的回答 |
|---|---|---|
| 描述符太多怎么办？ | 方差过滤 + 共线性剪枝 + RFE，29 → 10 维而精度不变 | — |
| 模型为什么这么判？ | Shapley 四公理 + TreeSHAP 全局/局部归因 | — |
| 二维表示的尽头在哪？ | SHAP 显示 `MolLogP` 一家独大，其余描述符贡献很薄 | 三维形状与晶格能根本不在 2D 描述符的表达范围内 |
| 缺的数据从哪来？ | — | PubChem PUG-REST / PUG-View 批量抓取 + 文本解析 |

### 下一讲预告

第五讲处理的始终是**单个有机分子**——它有明确的 SMILES、明确的分子量、明确的熔点。但如果研究对象是 $\text{BaTiO}_3$ 这样的**无机晶体**呢？它没有“分子”，只有周期性重复的晶胞；它的性质由空间群对称性、离子半径、配位环境共同决定。

下一讲 **`06 - Materials Informatics`（无机与晶体材料信息学）** 将换一套完全不同的工具：用 `pymatgen` 操作晶胞与空间群、从 Materials Project 数据库批量检索形成能与带隙、模拟 XRD 谱图、构建热力学凸包相图，并用元素组成描述符做高通量虚拟筛选。